# MFLI Lock-in Amplifier Test Notebook

Instantiates `instruments.mfli.MFLI`, connects, applies the default
demodulator/input/reference configuration, and reads back a single sample
and an averaged (noise-reduced) sample.

This class is built on `zhinst.core.ziDAQServer` rather than
`zhinst.toolkit` -- see the module docstring in `instruments/mfli.py` for
why. Several default values (input range, filter order, time constant,
device interface) were carried over from
`code_collection/MFLI_test.ipynb` and are flagged there as **CONFIRM ON
REAL HARDWARE**, since no MFLI has ever been connected from this
development environment.

**No real instrument is connected to this development machine.** Every
live-hardware call below is wrapped in `try`/`except` so this notebook
runs cleanly end-to-end and prints a clear "hardware not available"
message instead of raising, when run here. On the lab computer (with the
MFLI actually connected), the same cells should run against real
hardware.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from instruments.mfli import MFLI, MFLIError
from instruments import config

print(f"Default MFLI device: {config.MFLI_DEVICE_ID} at {config.MFLI_HOST}:{config.MFLI_PORT}")

## Connect

In [ ]:
mfli = MFLI()
mfli_connected = False

try:
    mfli.connect()
    mfli_connected = True
    print(f"Connected to MFLI {mfli.device_id} via {mfli.host}:{mfli.port}")
except Exception as exc:
    print(f"Hardware not available: could not connect to MFLI ({exc})")

## Apply default demodulator/input/reference configuration

In [ ]:
try:
    if not mfli_connected:
        raise RuntimeError("MFLI not connected")
    mfli.apply_default_configuration()
    print("Applied default demodulator/input/reference configuration.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Override a single setting (example: shorter time constant)

In [ ]:
try:
    if not mfli_connected:
        raise RuntimeError("MFLI not connected")
    mfli.configure_demod(0, enable=True, filter_order=4, time_constant=0.05)
    print("Demodulator 0 reconfigured with a shorter time constant.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Automatic phase adjustment

In [ ]:
try:
    if not mfli_connected:
        raise RuntimeError("MFLI not connected")
    mfli.auto_phase_adjust(0)
    print("Triggered automatic phase adjustment on demodulator 0.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Read a single sample and an averaged (noise-reduced) sample

In [ ]:
try:
    if not mfli_connected:
        raise RuntimeError("MFLI not connected")
    sample = mfli.read_sample(0)
    print("Single sample:", sample)
    averaged = mfli.read_averaged_sample(0, n_samples=10, delay=0.05)
    print("Averaged sample (noise-reduced):", averaged)
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Disconnect

In [ ]:
try:
    if mfli_connected:
        mfli.disconnect()
        print("Disconnected from MFLI.")
    else:
        print("Skipping disconnect: MFLI was never connected.")
except Exception as exc:
    print(f"Error during disconnect: {exc}")

## Context manager usage

In [ ]:
try:
    with MFLI() as lockin:
        print(f"Connected via context manager to {lockin.device_id}")
except Exception as exc:
    print(f"Hardware not available: {exc}")